# **Pré-processamento e Exportação do Dataset**

In [8]:

!pip install pynrrd
import os
import nrrd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from skimage.transform import resize

# 1. ESTRUTURA DE PASTAS E CONFIGURAÇÕES
pasta_base = "/content/drive/MyDrive/Segmentação_LCA"
pasta_saida = f"{pasta_base}/04_Dataset_Processado"
pasta_img_out = f"{pasta_saida}/Imagens"
pasta_mask_out = f"{pasta_saida}/Mascaras"

os.makedirs(pasta_img_out, exist_ok=True)
os.makedirs(pasta_mask_out, exist_ok=True)

ids_exames = [f"{n:04d}" for n in range(1, 26)]
tamanho_alvo = (256, 256)


cor_destaque = ListedColormap(['none', 'red'])
fig, axes = plt.subplots(len(ids_exames), 3, figsize=(10, 2.5 * len(ids_exames)), facecolor='#333333')

print("[LOG] Iniciando exportação do conjunto de dados final...\n")

# 2. PROCESSAMENTO E EXPORTAÇÃO
total_fatias = 0

for i, exam_id in enumerate(ids_exames):
    caminho_img = f"{pasta_base}/01_Imagens_Originais/image_LCA_{exam_id}.nrrd"
    caminho_mask = f"{pasta_base}/02_Mascaras_Referencia/mask_LCA_{exam_id}.nrrd"

    if not os.path.exists(caminho_img) or not os.path.exists(caminho_mask):
        continue

    print(f"A processar o exame LCA_{exam_id}...")
    img_vol, _ = nrrd.read(caminho_img)
    mask_vol, _ = nrrd.read(caminho_mask)

    fatias_com_lca = np.where(np.sum(mask_vol, axis=(0, 1)) > 0)[0]
    fatia_alvo = fatias_com_lca[len(fatias_com_lca) // 2] if len(fatias_com_lca) > 0 else img_vol.shape[2] // 2

    num_fatias = img_vol.shape[2]
    fatias_guardadas = 0

    for z in range(num_fatias):
        img_fatia = img_vol[:, :, z]
        mask_fatia = mask_vol[:, :, z]

        # Filtro: Avançar e ignorar as fatias sem a presença do LCA
        if np.sum(mask_fatia) == 0:
            continue

        # Rotação corretiva da geometria
        img_fatia = np.rot90(img_fatia, k=-1)
        mask_fatia = np.rot90(mask_fatia, k=-1)
        mask_fatia = (mask_fatia > 0).astype(np.uint8)

        # Normalização Min-Max (0.0 a 1.0)
        img_fatia_float = img_fatia.astype(np.float32)
        val_min, val_max = np.min(img_fatia_float), np.max(img_fatia_float)
        if val_max > val_min:
            img_norm = (img_fatia_float - val_min) / (val_max - val_min)
        else:
            img_norm = img_fatia_float

        # Redimensionamento para 256x256
        img_256 = resize(img_norm, tamanho_alvo, order=1, preserve_range=True, anti_aliasing=True).astype(np.float32)
        mask_256 = resize(mask_fatia, tamanho_alvo, order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)

        # Exportação (.npy)
        nome_ficheiro = f"LCA_{exam_id}_fatia_{z:03d}.npy"
        np.save(f"{pasta_img_out}/img_{nome_ficheiro}", img_256)
        np.save(f"{pasta_mask_out}/mask_{nome_ficheiro}", mask_256)

        fatias_guardadas += 1
        total_fatias += 1

        # 3. PLOTAGEM COMPARATIVA
        if z == fatia_alvo:
            mascara_transparente_256 = np.ma.masked_where(mask_256 == 0, mask_256)

            # Coluna 1: Imagem Original Bruta
            axes[i, 0].imshow(img_fatia, cmap='gray')
            axes[i, 0].set_title(f"LCA_{exam_id} - Bruto\nTamanho: {img_fatia.shape}\nTipo: {img_fatia.dtype} (Max: {np.max(img_fatia)})", color='white', fontsize=10)
            axes[i, 0].axis('off')

            # Coluna 2: Imagem Processada (Normalizada e 256x256)
            axes[i, 1].imshow(img_256, cmap='gray', vmin=0, vmax=1)
            axes[i, 1].set_title(f"LCA_{exam_id} - Processado\nTamanho: {img_256.shape}\nTipo: float32 (Max: {np.max(img_256):.1f})", color='#00ffcc', fontsize=10)
            axes[i, 1].axis('off')

            # Coluna 3: Validação da Sobreposição pós-resize
            axes[i, 2].imshow(img_256, cmap='gray', vmin=0, vmax=1)
            axes[i, 2].imshow(mascara_transparente_256, cmap=cor_destaque, vmin=0, vmax=1, alpha=0.7)
            axes[i, 2].set_title(f"LCA_{exam_id} - Sobreposição (256x256)", color='white', fontsize=10)
            axes[i, 2].axis('off')

    print(f" -> {fatias_guardadas} fatias com LCA processadas e guardadas.")

print(f"\n[SUCESSO] Operação concluída! {total_fatias} matrizes .npy guardadas no Drive.")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.